# 02 — Silver validation: eventstream 4-gate quality contract

| Field | Value |
| ----- | ----- |
| **Sprint** | Sprint 09 v2 — T2.7 |
| **Layer** | `silver/eventstream/` |
| **Source** | `Tables/bronze/eventstream/<eventKind>/` (Delta, appended by `01_bronze_eventstream.ipynb`) |
| **Target** | `Tables/silver/eventstream/<eventKind>/` (Delta, validated) + `Tables/silver/eventstream/_quarantine/<eventKind>/` (rejected rows) |
| **Governance** | [ADR-0015](../../../docs/adr/0015-skip-sql-for-mvp-demo.md), [ADR-0016 Gate 2 — Ingestion PHI regex](../../../docs/adr/0016-no-phi-in-mvp-demo-scope.md), [ADR-0013 dual-mode residency](../../../docs/adr/0013-temporary-us-region-demo-scope.md) |
| **Design spec** | [§4.6 validation gates](../../../docs/superpowers/specs/2026-07-02-sprint-09-v2-refinement-design.md) |

## Purpose

Apply the 4-gate quality contract to every bronze eventstream partition. Rows
that violate a gate are **quarantined** to `silver/eventstream/_quarantine/<eventKind>/`
(design spec §4.6 "Reject row → quarantine") rather than aborting the whole
notebook — streaming ingestion must degrade gracefully when a single malformed
envelope arrives. A **PHI regex hit still aborts** per ADR-0016 (any positive
match on a synthetic dataset is a bug that must be triaged, not accumulated).

## The 4 validation gates (design spec §4.6)

| # | Gate | Contract | Fail action |
| - | ---- | -------- | ----------- |
| 1 | Schema | Payload conforms to `dc-*-v1.schema.json` (via `SCHEMA_MAP` below) | Reject row → quarantine |
| 2 | PHI regex sweep | No **source** cell matches email / phone / DOB / CH AHV-13 patterns (ADR-0016 Gate 2, same regex bundle as `02_silver_master_data.ipynb`) | Reject rows + alert; **abort** on any hit |
| 3 | FK integrity | For `bed.assigned` + `discharge.scored` + `discharge.recommended`: referenced `encounterId` exists in `silver/eventstream/encounter.admitted/` or `silver/eventstream/encounter.transitioned/` | Reject row → quarantine; abort if orphan share > 5% (design spec §4.6) |
| 4 | Residency | `_residency_tag ∈ {CH-North, US-West}` (RB-01 dual-mode) | Reject row → quarantine |

PHI Gate 2 is deterministic (pure regex, no LLM) and shares its pattern bundle
with `02_silver_master_data.ipynb` — cross-notebook consistency for the ADR-0016
gate. The self-test cell at the end seeds a bogus `encounter.admitted` envelope
with a fake email in a payload field and asserts the gate rejects it (abort).

In [ ]:
# Parameters (Fabric injects overrides via the papermill-compatible 'parameters' tag).
target_lakehouse = 'lh_ihzhhpf_sit'
bronze_root = 'Tables/bronze/eventstream'
silver_root = 'Tables/silver/eventstream'
quarantine_root = 'Tables/silver/eventstream/_quarantine'
run_id = 'run-manual-local'
orphan_fk_threshold = 0.05                 # 5% per design spec §4.6
default_residency_tag = 'US-West'          # demo scope per ADR-0013 (westus2 carve-out until 2026-09-30)
schema_dir = 'Files/schema'                # lakehouse-relative folder containing dc-*-v1.schema.json copies (mirrored from repo data/synthetic/schema/)

In [ ]:
# eventKind → payload schema map (design spec §4.3 + §4.6). Kinds without a dedicated
# JSON schema in data/synthetic/schema/ use a permissive validator that only asserts
# payload is a JSON object — they are still subject to Gates 2 (PHI), 3 (FK), 4 (residency).
SCHEMA_MAP = {
    'encounter.admitted':      'dc-demand-encounter-v1.schema.json',
    'encounter.transitioned':  'dc-demand-encounter-v1.schema.json',   # same entity, transition sub-record inside payload
    'bed.state_changed':       None,                                    # no dedicated schema in repo — permissive validation
    'bed.assigned':            'dc-match-recommendation-v1.schema.json',
    'forecast.published':      'dc-demand-forecast-v1.schema.json',
    'discharge.scored':        'dc-discharge-score-v1.schema.json',
    'discharge.recommended':   'dc-discharge-recommendation-v1.schema.json',
}
EVENT_KINDS = list(SCHEMA_MAP.keys())

# Kinds whose payload MUST reference an existing encounterId (Gate 3).
FK_ENCOUNTER_KINDS = {'bed.assigned', 'discharge.scored', 'discharge.recommended'}

In [ ]:
# PHI regex catalogue — identical to 02_silver_master_data.ipynb per ADR-0016 Gate 2.
# Cross-notebook consistency is a hard contract: any change must be applied in both places.
import re

PHI_PATTERNS = {
    'email':      re.compile(r'[\w.+-]+@[\w-]+\.[\w.-]+'),
    'phone':      re.compile(r'\+?\d[\d\s().-]{6,}'),
    'dob':        re.compile(r'\d{4}-\d{2}-\d{2}'),
    'ch_ahv_13':  re.compile(r'756\.\d{4}\.\d{4}\.\d{2}'),
}
ALLOWED_RESIDENCY_TAGS = {'CH-North', 'US-West'}

In [ ]:
import json
from datetime import datetime, timezone
from pyspark.sql import DataFrame, Row, functions as F, types as T

try:
    import jsonschema  # Fabric Spark runtime ships jsonschema
    _HAS_JSONSCHEMA = True
except ImportError:
    _HAS_JSONSCHEMA = False

class GateAbort(Exception):
    """Raised when a gate demands the whole notebook aborts (PHI hit)."""

def _now_iso() -> str:
    return datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

def _load_schema(schema_filename: str):
    if schema_filename is None:
        return None
    # Fabric Spark reads a Files-relative path via spark.read.text and reassembles.
    raw = spark.read.text(f'{schema_dir}/{schema_filename}').agg(F.concat_ws('\n', F.collect_list('value')).alias('s')).collect()[0]['s']
    return json.loads(raw)

In [ ]:
# ----- Gate 1: schema validation -----
def gate1_schema(df: DataFrame, eventKind: str, schema_doc):
    """Return (accepted_df, rejected_df) partitioning by schema conformance.

    If jsonschema is unavailable OR schema_doc is None, applies a permissive rule:
    payload must exist and be parseable as JSON object.
    """
    # Bring the payload into a Python-side check via a UDF returning a boolean pass flag.
    if _HAS_JSONSCHEMA and schema_doc is not None:
        validator = jsonschema.Draft7Validator(schema_doc)
        def _validate(payload_str):
            if payload_str is None:
                return False
            try:
                obj = json.loads(payload_str) if isinstance(payload_str, str) else payload_str
            except Exception:
                return False
            return validator.is_valid(obj)
    else:
        def _validate(payload_str):
            if payload_str is None:
                return False
            try:
                obj = json.loads(payload_str) if isinstance(payload_str, str) else payload_str
                return isinstance(obj, dict)
            except Exception:
                return False

    # Payload may already be a struct (Eventstream materialisation) or a JSON string.
    # Normalise to string for the UDF.
    if 'payload' not in df.columns:
        # No payload column at all — whole batch is malformed.
        return df.limit(0), df
    payload_field = df.schema['payload']
    if isinstance(payload_field.dataType, T.StringType):
        payload_str = F.col('payload')
    else:
        payload_str = F.to_json(F.col('payload'))
    validate_udf = F.udf(_validate, T.BooleanType())
    tagged = df.withColumn('_schema_ok', validate_udf(payload_str))
    accepted = tagged.filter(F.col('_schema_ok')).drop('_schema_ok')
    rejected = tagged.filter(~F.col('_schema_ok')).drop('_schema_ok') \
                     .withColumn('_reject_reason', F.lit(f'gate1_schema:{eventKind}'))
    return accepted, rejected

# ----- Gate 2: PHI regex sweep (ADR-0016) -----
def gate2_phi_scan(df: DataFrame, eventKind: str):
    """Scan every source string column against PHI_PATTERNS. Abort on any hit.

    Governance/lineage columns (leading `_`) are excluded so the DOB regex does not
    false-positive on the `_lineage_ref` ISO timestamp. Matches shared discipline
    with 02_silver_master_data.ipynb Gate 3.
    """
    string_cols = [f.name for f in df.schema.fields
                   if isinstance(f.dataType, T.StringType) and not f.name.startswith('_')]
    if not string_cols:
        return df, df.limit(0)
    hit_expr = None
    for _pname, pattern in PHI_PATTERNS.items():
        for col in string_cols:
            match = F.col(col).rlike(pattern.pattern)
            hit_expr = match if hit_expr is None else (hit_expr | match)
    rejected = df.filter(hit_expr)
    clean = df.filter(~hit_expr)
    n_rej = rejected.count()
    if n_rej > 0:
        print(f'ALERT [{eventKind}] gate 2 PHI regex sweep rejected {n_rej} row(s) at {_now_iso()}')
        raise GateAbort(f'{eventKind}: gate 2 PHI regex sweep failed — {n_rej} row(s) matched PHI patterns; run aborted')
    return clean, rejected

# ----- Gate 3: FK integrity for encounter-referencing kinds -----
def gate3_fk_encounter(df: DataFrame, eventKind: str, encounter_id_set: DataFrame):
    """Verify `payload.encounterId` exists in the union of silver encounter tables.

    Rows with an orphan encounterId are quarantined; if the orphan share exceeds
    orphan_fk_threshold (5% per design spec §4.6) the notebook aborts.
    """
    if eventKind not in FK_ENCOUNTER_KINDS:
        return df, df.limit(0)
    # Extract encounterId from payload (payload may be struct or JSON string).
    payload_field = df.schema['payload']
    if isinstance(payload_field.dataType, T.StringType):
        with_enc = df.withColumn('__encounterId', F.get_json_object(F.col('payload'), '$.encounterId'))
    else:
        with_enc = df.withColumn('__encounterId', F.col('payload.encounterId'))
    if encounter_id_set is None:
        # No encounter silver yet — quarantine everything to avoid propagating unverified FK.
        rejected = with_enc.withColumn('_reject_reason', F.lit(f'gate3_fk:{eventKind}:no_encounter_silver')).drop('__encounterId')
        return df.limit(0), rejected
    total = with_enc.count() or 1
    joined = with_enc.join(encounter_id_set, with_enc['__encounterId'] == encounter_id_set['__encounter_key'], 'left')
    accepted = joined.filter(F.col('__encounter_key').isNotNull()) \
                     .drop('__encounterId', '__encounter_key')
    orphans = joined.filter(F.col('__encounter_key').isNull()) \
                    .drop('__encounterId', '__encounter_key') \
                    .withColumn('_reject_reason', F.lit(f'gate3_fk:{eventKind}:orphan_encounterId'))
    n_orphan = orphans.count()
    if total > 0 and (n_orphan / total) > orphan_fk_threshold:
        raise GateAbort(f'{eventKind}: gate 3 FK integrity — {n_orphan}/{total} ({n_orphan/total:.1%}) orphan encounterId > {orphan_fk_threshold:.0%} threshold')
    if n_orphan > 0:
        print(f'WARN [{eventKind}] gate 3 FK integrity quarantined {n_orphan}/{total} orphan row(s)')
    return accepted, orphans

# ----- Gate 4: residency -----
def gate4_residency(df: DataFrame, eventKind: str):
    tagged = df.withColumn('_residency_tag', F.lit(default_residency_tag)) if '_residency_tag' not in df.columns \
             else df.withColumn('_residency_tag', F.coalesce(F.col('_residency_tag'), F.lit(default_residency_tag)))
    accepted = tagged.filter(F.col('_residency_tag').isin(list(ALLOWED_RESIDENCY_TAGS)))
    rejected = tagged.filter(~F.col('_residency_tag').isin(list(ALLOWED_RESIDENCY_TAGS))) \
                     .withColumn('_reject_reason', F.lit(f'gate4_residency:{eventKind}'))
    return accepted, rejected

In [ ]:
def _write_quarantine(df: DataFrame, eventKind: str):
    if df.rdd.isEmpty():
        return 0
    n = df.count()
    tgt = f'{quarantine_root}/{eventKind}'
    # Quarantine schemas evolve as new reject_reason cols appear — allow schema merge.
    (df.write.format('delta').mode('append').option('mergeSchema', 'true').save(tgt))
    return n

def _write_silver(df: DataFrame, eventKind: str):
    tgt = f'{silver_root}/{eventKind}'
    (df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save(tgt))
    return df.count()

In [ ]:
def _build_encounter_id_set():
    """Union of encounterIds across the two encounter silver partitions, if present."""
    frames = []
    for kind in ('encounter.admitted', 'encounter.transitioned'):
        try:
            df = spark.read.format('delta').load(f'{silver_root}/{kind}')
        except Exception:
            continue
        if 'payload' not in df.columns:
            continue
        payload_field = df.schema['payload']
        if isinstance(payload_field.dataType, T.StringType):
            enc = df.select(F.get_json_object(F.col('payload'), '$.encounterId').alias('__encounter_key'))
        else:
            enc = df.select(F.col('payload.encounterId').alias('__encounter_key'))
        frames.append(enc.filter(F.col('__encounter_key').isNotNull()).distinct())
    if not frames:
        return None
    from functools import reduce
    return reduce(lambda a, b: a.unionByName(b), frames).distinct()

def promote_event_kind(eventKind: str, encounter_id_set):
    src = f'{bronze_root}/{eventKind}'
    try:
        df = spark.read.format('delta').load(src)
    except Exception as exc:
        print(f'SKIP [{eventKind}] no bronze partition yet ({exc})')
        return {'eventKind': eventKind, 'bronze_rows': 0, 'silver_rows': 0, 'quarantine_rows': 0, 'skipped': True}
    n_bronze = df.count()
    if n_bronze == 0:
        return {'eventKind': eventKind, 'bronze_rows': 0, 'silver_rows': 0, 'quarantine_rows': 0, 'skipped': False}

    schema_doc = _load_schema(SCHEMA_MAP[eventKind])
    accepted, rej_schema = gate1_schema(df, eventKind, schema_doc)
    accepted, _rej_phi = gate2_phi_scan(accepted, eventKind)  # aborts on hit
    accepted, rej_fk = gate3_fk_encounter(accepted, eventKind, encounter_id_set)
    accepted, rej_res = gate4_residency(accepted, eventKind)

    q_total = 0
    for rej in (rej_schema, rej_fk, rej_res):
        q_total += _write_quarantine(rej, eventKind)
    n_silver = _write_silver(accepted, eventKind)
    return {'eventKind': eventKind, 'bronze_rows': n_bronze, 'silver_rows': n_silver, 'quarantine_rows': q_total, 'skipped': False}

In [ ]:
# Promote in dependency order: encounter kinds first (they build the FK set), then the rest.
ORDER = [
    'encounter.admitted', 'encounter.transitioned',  # build FK set
    'bed.state_changed',
    'bed.assigned',
    'forecast.published',
    'discharge.scored', 'discharge.recommended',
]

results = []
# First pass: promote both encounter kinds so silver has them.
for kind in ('encounter.admitted', 'encounter.transitioned'):
    results.append(promote_event_kind(kind, encounter_id_set=None))  # encounter kinds don't need FK check
encounter_id_set = _build_encounter_id_set()
# Second pass: promote the rest with the FK set available.
for kind in ORDER:
    if kind in ('encounter.admitted', 'encounter.transitioned'):
        continue
    results.append(promote_event_kind(kind, encounter_id_set=encounter_id_set))

print(f'Silver eventstream promotion summary (run_id={run_id})')
print('-' * 88)
for r in results:
    print(f"  {r['eventKind']:<28s} bronze={r['bronze_rows']:<6d} silver={r['silver_rows']:<6d} quarantine={r['quarantine_rows']:<6d} skipped={r['skipped']}")

## PHI regex gate — deterministic self-test

Seed one bogus `encounter.admitted` envelope with a fake email injected into a
payload field (`firstName`) and confirm Gate 2 raises `GateAbort`. Mirrors the
self-test discipline in `02_silver_master_data.ipynb` — cross-notebook parity
for the ADR-0016 gate.

**Expected output**: an `ALERT` line naming `encounter.admitted__phi_test`,
`GateAbort` raised and captured, then `PHI GATE TEST: PASSED`.

In [ ]:
test_rows = [
    Row(eventKind='encounter.admitted', eventId='e_ok',  hospitalId='H_USZ',
        simRunId='sr_test', seed=42, simulatedAt='2027-01-15T10:00:00Z', emittedAt='2026-07-03T00:00:00Z',
        payload='{"encounterId": "ENC_OK", "firstName": "Anonymised"}',
        _lineage_ref='test:2026-07-03T00:00:00Z'),
    Row(eventKind='encounter.admitted', eventId='e_bad', hospitalId='H_USZ',
        simRunId='sr_test', seed=42, simulatedAt='2027-01-15T10:01:00Z', emittedAt='2026-07-03T00:00:00Z',
        payload='{"encounterId": "ENC_BAD", "firstName": "leak@example.com"}',  # fake email must trip PHI regex
        _lineage_ref='test:2026-07-03T00:00:00Z'),
]
test_df = spark.createDataFrame(test_rows)

gate_raised = False
gate_message = None
try:
    gate2_phi_scan(test_df, 'encounter.admitted__phi_test')
except GateAbort as exc:
    gate_raised = True
    gate_message = str(exc)

assert gate_raised, 'PHI gate did NOT raise on synthetic bogus envelope — regression detected'
assert 'encounter.admitted__phi_test' in gate_message, 'PHI gate raised but eventKind label missing from error'
assert '1 row(s) matched' in gate_message, f'Expected exactly 1 rejected row; got: {gate_message}'
print('PHI GATE TEST: PASSED')
print(f'  captured: {gate_message}')